# ============================================================================
# HARMONY-CORRECTED PCA EMBEDDING EXTRACTION
# Input: PNG patches from Kaggle dataset (barcode parsed from filename)
# Output: (50,) Harmony PCA embeddings per patch
# ============================================================================

In [1]:
install.packages(c("Seurat", "Matrix", "harmony", "reticulate"),
                 repos = "https://cloud.r-project.org", quiet = TRUE)

cat("Seurat :", as.character(packageVersion("Seurat")), "\n")
cat("harmony:", as.character(packageVersion("harmony")), "\n")
cat("Matrix :", as.character(packageVersion("Matrix")), "\n")

Seurat : 5.5.0 


harmony: 2.0.3 


Matrix : 1.7.5 


In [2]:
library(Seurat)
library(Matrix)
library(harmony)
library(reticulate)

CONFIG <- list(
    rds_root         = "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/ST/ST",
    png_root         = "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/.png patches/.png patches",
    output_root      = "/kaggle/working/Features/HarmonyPCA",
    samples          = c("IU_PDA_HM11", "IU_PDA_HM13", "IU_PDA_T1",
                         "IU_PDA_T11",  "IU_PDA_T3",   "IU_PDA_T4"),
    n_hvgs           = 3000,
    n_pcs            = 50,
    harmony_theta    = 2.0,
    harmony_max_iter = 20,
    forced_genes_path = NULL   # path to .txt with one gene per line, or NULL
)

cat("RDS root     :", CONFIG$rds_root,    "\n")
cat("Output root  :", CONFIG$output_root, "\n")
cat("Samples      :", paste(CONFIG$samples, collapse=", "), "\n")
cat("Embedding dim:", CONFIG$n_pcs, "\n")

Loading required package: SeuratObject



Loading required package: sp



‘SeuratObject’ was built with package ‘Matrix’ 1.7.1 but the current
version is 1.7.5; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed




Attaching package: ‘SeuratObject’




The following objects are masked from ‘package:base’:

    intersect, t




...

• Get help: Visit the website at https://korsunskylab.github.io/harmony2/ and
report issues on https://github.com/immunogenomics/harmony/issues


RDS root     : /kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/ST/ST 


Output root  : /kaggle/working/Features/HarmonyPCA 


Samples      : IU_PDA_HM11, IU_PDA_HM13, IU_PDA_T1, IU_PDA_T11, IU_PDA_T3, IU_PDA_T4 


Embedding dim: 50 


In [3]:
cat(strrep("=", 70), "\n")
cat("LOADING SEURAT OBJECTS\n")
cat(strrep("=", 70), "\n")

count_list <- list()

for (sample_name in CONFIG$samples) {

    rds_path <- file.path(CONFIG$rds_root, paste0(sample_name, ".rds"))

    if (!file.exists(rds_path)) {
        cat("  WARNING: Not found —", rds_path, "\n")
        next
    }

    cat("\nLoading:", sample_name, "\n")
    obj <- readRDS(rds_path)

    # Extract raw counts — handles Seurat v4 and v5
    counts_mat <- tryCatch({
        GetAssayData(obj, assay = "Spatial", slot = "counts")        # Seurat v4
    }, error = function(e) {
        tryCatch({
            GetAssayData(obj, assay = "Spatial", layer = "counts")   # Seurat v5
        }, error = function(e2) {
            obj[["Spatial"]]@counts                                  # direct slot
        })
    })

    cat("  Dims (genes x barcodes):", nrow(counts_mat), "x", ncol(counts_mat), "\n")
    count_list[[sample_name]] <- counts_mat

    rm(obj); gc()
}

cat("\n", strrep("=", 70), "\n", sep="")
cat("Loaded", length(count_list), "/", length(CONFIG$samples), "samples\n")

LOADING SEURAT OBJECTS



Loading: IU_PDA_HM11 


Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.”


  Dims (genes x barcodes): 17893 x 3931 

Loading: IU_PDA_HM13 
  Dims (genes x barcodes): 17893 x 2182 

Loading: IU_PDA_T1 
  Dims (genes x barcodes): 17893 x 3530 

Loading: IU_PDA_T11 
  Dims (genes x barcodes): 17893 x 2777 

Loading: IU_PDA_T3 
  Dims (genes x barcodes): 17893 x 4354 

Loading: IU_PDA_T4 
  Dims (genes x barcodes): 17893 x 3621 


Loaded 6 / 6 samples


In [4]:
cat(strrep("=", 70), "\n")
cat("PREPROCESSING + HARMONY PCA\n")
cat(strrep("=", 70), "\n\n")

# ---- 1. Merge on common genes ----
cat("Merging count matrices...\n")
common_genes  <- Reduce(intersect, lapply(count_list, rownames))
cat("  Common genes:", length(common_genes), "\n")

merged_counts <- do.call(cbind, lapply(names(count_list), function(s) {
    count_list[[s]][common_genes, ]
}))

sample_labels <- unlist(lapply(names(count_list), function(s) {
    rep(s, ncol(count_list[[s]]))
}))

cat("  Merged:", nrow(merged_counts), "genes x", ncol(merged_counts), "barcodes\n\n")

# ---- 2. Seurat object ----
cat("Creating Seurat object...\n")
seurat_merged          <- CreateSeuratObject(counts = merged_counts)
seurat_merged$sample   <- sample_labels
rm(merged_counts); gc()

# ---- 3. Normalize ----
cat("Normalizing...\n")
seurat_merged <- NormalizeData(seurat_merged,
                               normalization.method = "LogNormalize",
                               scale.factor = 10000, verbose = FALSE)

# ---- 4. HVG ----
cat("Finding", CONFIG$n_hvgs, "HVGs...\n")
seurat_merged <- FindVariableFeatures(seurat_merged,
                                      selection.method = "vst",
                                      nfeatures = CONFIG$n_hvgs,
                                      verbose = FALSE)
hvgs <- VariableFeatures(seurat_merged)

# Force marker genes if provided
if (!is.null(CONFIG$forced_genes_path) && file.exists(CONFIG$forced_genes_path)) {
    forced_genes   <- readLines(CONFIG$forced_genes_path)
    forced_present <- forced_genes[forced_genes %in% rownames(seurat_merged)]
    forced_missing <- forced_genes[!forced_genes %in% rownames(seurat_merged)]
    if (length(forced_missing) > 0)
        cat("  NOTE:", length(forced_missing), "forced genes absent:",
            paste(forced_missing, collapse=", "), "\n")
    hvgs <- union(hvgs, forced_present)
    cat("  Forced", length(forced_present), "marker genes into HVG set\n")
}
cat("  Using", length(hvgs), "HVGs\n\n")

# ---- 5. Scale ----
cat("Scaling...\n")
seurat_merged <- ScaleData(seurat_merged, features = hvgs, verbose = FALSE)

# ---- 6. PCA ----
cat("Running PCA (", CONFIG$n_pcs, "dims)...\n")
seurat_merged <- RunPCA(seurat_merged, features = hvgs,
                        npcs = CONFIG$n_pcs, verbose = FALSE)
cat("  PCA done\n\n")

# ---- 7. Harmony ----
cat("Running Harmony...\n")
seurat_merged <- RunHarmony(
    seurat_merged,
    group.by.vars = "sample",
    dims.use      = 1:CONFIG$n_pcs,
    theta         = CONFIG$harmony_theta,
    verbose       = FALSE
)

harmony_emb <- Embeddings(seurat_merged, reduction = "harmony")
cat("  Harmony embedding:", nrow(harmony_emb), "x", ncol(harmony_emb), "\n\n")
cat(strrep("=", 70), "\n")
cat("Preprocessing complete\n")

PREPROCESSING + HARMONY PCA


Merging count matrices...


  Common genes: 17893 


  Merged: 17893 genes x 20395 barcodes



Creating Seurat object...


...

Running PCA ( 50 dims)...


  PCA done



Running Harmony...


  Harmony embedding: 20395 x 50 



Preprocessing complete


In [5]:
# ---- Build barcode lookup from coordinates file ----
coords <- read.csv("/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/spot_spatial_coordinates.csv")
coords$rc_key <- paste(coords$image, coords$row, coords$col, sep="_")
rc_to_barcode <- setNames(coords$spot_barcode, coords$rc_key)
cat("Lookup ready:", length(rc_to_barcode), "entries\n")

Lookup ready: 91496 entries


In [6]:
cat(strrep("=", 70), "\n")
cat("SAVING PER-PATCH EMBEDDINGS\n")
cat(strrep("=", 70), "\n\n")

all_barcodes <- rownames(harmony_emb)

parse_row_col <- function(filename) {
    # Extract row and col from filename pattern: <SAMPLE>_patch-XXXXXX_<ROW>_<COL>.png
    stem  <- tools::file_path_sans_ext(basename(filename))
    parts <- strsplit(stem, "_")[[1]]
    n     <- length(parts)
    # last two tokens are row and col
    list(row = as.integer(parts[n-1]), col = as.integer(parts[n]))
}

total_saved   <- 0
total_missing <- 0
total_failed  <- 0

for (sample_name in CONFIG$samples) {

    cat("\n", strrep("=", 70), "\n", sep="")
    cat("Processing:", sample_name, "\n")
    cat(strrep("=", 70), "\n")

    sample_png_dir <- file.path(CONFIG$png_root, sample_name)
    if (!dir.exists(sample_png_dir)) {
        cat("  PNG directory not found:", sample_png_dir, "\n")
        next
    }

    patches <- list.files(sample_png_dir, pattern = "\\.png$", full.names = TRUE)
    cat("  Found", length(patches), "patches\n")
    if (length(patches) == 0) next

    out_dir <- file.path(CONFIG$output_root, sample_name)
    dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

    # Resume: skip already saved
    existing   <- tools::file_path_sans_ext(list.files(out_dir, pattern = "\\.csv$"))
    to_process <- patches[
        !tools::file_path_sans_ext(basename(patches)) %in% existing
    ]

    if (length(to_process) == 0) {
        cat("  All", length(existing), "patches already processed\n")
        next
    }
    cat("  Processing", length(to_process),
        "new patches (existing:", length(existing), ")\n\n")

    saved   <- 0
    missing <- 0
    failed  <- 0

    for (patch_path in to_process) {
        patch_name <- tools::file_path_sans_ext(basename(patch_path))
        rc         <- parse_row_col(patch_path)

        # Step 1: row+col -> barcode via coords lookup
        rc_key  <- paste(sample_name, rc$row, rc$col, sep="_")
        barcode <- rc_to_barcode[rc_key]

        if (is.na(barcode)) {
            missing <- missing + 1
            next
        }

        # Step 2: barcode -> harmony embedding row
        row_idx <- match(barcode, all_barcodes)

        if (is.na(row_idx)) {
            missing <- missing + 1
            next
        }

        save_path <- file.path(out_dir, paste0(patch_name, ".csv"))

        tryCatch({
            emb_vec <- as.numeric(harmony_emb[row_idx, ])   # (50,)
            write.csv(
                data.frame(t(emb_vec)),
                save_path,
                row.names = FALSE
            )
            saved <- saved + 1
        }, error = function(e) {
            cat("  Error on", patch_name, ":", conditionMessage(e), "\n")
            failed <<- failed + 1
        })
    }

    cat("  Complete:", saved, "/", length(to_process), "saved\n")
    if (missing > 0) cat("  ", missing, "patches skipped (barcode not found)\n")
    if (failed  > 0) cat("  ", failed,  "patches failed\n")

    total_saved   <- total_saved   + saved
    total_missing <- total_missing + missing
    total_failed  <- total_failed  + failed
}

cat("\n", strrep("=", 70), "\n", sep="")
cat("TOTAL SAVED  :", total_saved,   "\n")
cat("TOTAL MISSING:", total_missing, "\n")
cat("TOTAL FAILED :", total_failed,  "\n")
cat("Output       :", CONFIG$output_root, "\n")

SAVING PER-PATCH EMBEDDINGS



Processing: IU_PDA_HM11 
  Found 3931 patches
  Processing 3931 new patches (existing: 0 )

  Complete: 3931 / 3931 saved

Processing: IU_PDA_HM13 
  Found 2182 patches
  Processing 2182 new patches (existing: 0 )

  Complete: 2182 / 2182 saved

Processing: IU_PDA_T1 
  Found 3530 patches
  Processing 3530 new patches (existing: 0 )

  Complete: 3530 / 3530 saved

Processing: IU_PDA_T11 
  Found 2777 patches
  Processing 2777 new patches (existing: 0 )

  Complete: 2777 / 2777 saved

Processing: IU_PDA_T3 
  Found 4354 patches
  Processing 4354 new patches (existing: 0 )

  Complete: 4354 / 4354 saved

Processing: IU_PDA_T4 
  Found 3621 patches
  Processing 3621 new patches (existing: 0 )

  Complete: 3621 / 3621 saved


TOTAL SAVED  : 20395 


TOTAL MISSING: 0 


TOTAL FAILED : 0 


Output       : /kaggle/working/Features/HarmonyPCA 


In [7]:
cat(strrep("=", 70), "\n")
cat("SUMMARY\n")
cat(strrep("=", 70), "\n\n")

total <- 0
for (sample_name in CONFIG$samples) {
    sample_dir <- file.path(CONFIG$output_root, sample_name)
    if (dir.exists(sample_dir)) {
        count <- length(list.files(sample_dir, pattern = "\\.csv$"))
        cat("  •", sample_name, ":", count, "embeddings\n")
        total <- total + count
    }
}
cat("\nTotal embeddings saved:", total, "\n")
cat("All outputs in        :", CONFIG$output_root, "\n")

SUMMARY


  • IU_PDA_HM11 : 3931 embeddings
  • IU_PDA_HM13 : 2182 embeddings
  • IU_PDA_T1 : 3530 embeddings
  • IU_PDA_T11 : 2777 embeddings
  • IU_PDA_T3 : 4354 embeddings
  • IU_PDA_T4 : 3621 embeddings



Total embeddings saved: 20395 


All outputs in        : /kaggle/working/Features/HarmonyPCA 


In [8]:
# Run this at the end of your R notebook before saving the version
# Kaggle saves everything in /kaggle/working/ as output
cat("Files in output:\n")
system("find /kaggle/working/Features/HarmonyPCA -name '*.csv' | head -20")

Files in output:
